In [0]:
from pyspark.sql.functions import *
import json

In [0]:
for job in (30001, 30002, 30003, 30004, 30005):
    with open(f"/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/gold/{job}.json", "r") as job:
        job_details = json.load(job)
        job_details["partition"] = spark.range(1).select(date_format(current_timestamp(), "yyyyMMddHHmmssSSS").cast("bigint").alias('partition')).collect()[0]['partition']
        dbutils.notebook.run("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/ETL/scd_type_1", 500, {"job_parameters" : json.dumps(job_details)})

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_geolocation as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.geolocation where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_orders as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.orders where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_order_items as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.order_items where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_order_payments as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.order_payments where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_order_reviews as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.order_reviews where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_customers as (
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.customers where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_products as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.products where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_product_category_name_translation as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.product_category_name_translation where bad_record_flag = false
)

In [0]:
%sql
create or replace view ecommerce.silver.vw_cln_ltst_sellers as(
    select * except (bad_record_flag, bad_record_reason) from ecommerce.silver.sellers where bad_record_flag = false
)

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.silver.vw_orders_enriched AS

WITH geo_agg AS (
    -- One row per zip code with averaged lat/lng
    SELECT
        geolocation_zip_code_prefix,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng,
        FIRST(geolocation_city) AS city,
        FIRST(geolocation_state) AS state
    FROM ecommerce.silver.vw_cln_ltst_geolocation
    GROUP BY geolocation_zip_code_prefix
),

customers_geo AS (
    SELECT
        c.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        g.lat,
        g.lng
    FROM ecommerce.silver.vw_cln_ltst_customers c
    LEFT JOIN geo_agg g
        ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    c.customer_city,
    c.lat,
    c.lng,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    -- Delivery metrics
    DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) AS delivery_delay_days,
    DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) AS actual_delivery_days,
    CASE WHEN DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END AS is_late,

    -- Approval speed
    ((UNIX_TIMESTAMP(o.order_approved_at) - UNIX_TIMESTAMP(o.order_purchase_timestamp)) / 3600)::decimal(38, 18) AS approval_time_hrs,

    -- Time dimensions
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS order_purchase_month,
    DAYOFWEEK(o.order_purchase_timestamp) AS order_purchase_dayofweek,
    HOUR(o.order_purchase_timestamp) AS order_purchase_hour,

    -- Status flag
    CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END AS is_delivered

FROM ecommerce.silver.vw_cln_ltst_orders o
LEFT JOIN customers_geo c ON o.customer_id = c.customer_id;

In [0]:
%sql
select * from ecommerce.silver.vw_orders_enriched

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.silver.vw_order_items_enriched AS

WITH seller_stats AS (
    -- Pre-aggregate seller metrics
    SELECT
        seller_id,
        COUNT(order_id) AS seller_total_orders,
        AVG(price)::decimal(38, 18) AS seller_avg_price,
        SUM(price)::decimal(38, 18) AS seller_total_revenue
    FROM ecommerce.silver.vw_cln_ltst_order_items
    GROUP BY seller_id
),

products_en AS (
    SELECT
        p.product_id,
        p.product_category_name,
        COALESCE(t.product_category_name_english, p.product_category_name) AS category_english,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm
    FROM ecommerce.silver.vw_cln_ltst_products p
    LEFT JOIN ecommerce.silver.vw_cln_ltst_product_category_name_translation t
        ON p.product_category_name = t.product_category_name
)

SELECT
    i.order_id,
    i.order_item_id,
    i.product_id,
    i.seller_id,
    i.shipping_limit_date,
    i.price::decimal(38, 18),
    i.freight_value::decimal(38, 18),

    -- Product info
    p.category_english,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,
    (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS product_volume_cm3,

    -- Seller info
    s.seller_city,
    s.seller_state,
    st.seller_total_orders,
    st.seller_avg_price,
    st.seller_total_revenue,

    -- Revenue metrics
    (i.price + i.freight_value)::decimal(38, 18) AS item_total_revenue,
    (ROUND(try_divide(i.freight_value, (i.price + i.freight_value) * 100), 5))::decimal(38, 18) AS freight_pct_of_revenue,

    -- Item flags
    (CASE WHEN i.price >= 200 THEN 1 ELSE 0 END) AS is_high_value_item

FROM ecommerce.silver.vw_cln_ltst_order_items i
LEFT JOIN products_en  p  ON i.product_id = p.product_id
LEFT JOIN ecommerce.silver.vw_cln_ltst_sellers s  ON i.seller_id = s.seller_id
LEFT JOIN seller_stats st ON i.seller_id = st.seller_id;

In [0]:
%sql
select * from ecommerce.silver.vw_order_items_enriched

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.silver.vw_payments_agg AS

SELECT
    order_id,
    SUM(payment_value)::decimal(38, 18) AS total_payment_value,
    COUNT(payment_sequential) AS total_payment_installments,
    COUNT(DISTINCT payment_type) AS num_payment_types_used,
    MAX(payment_installments) AS max_installments,

    -- Payment type breakdown
    SUM(CASE WHEN payment_type = 'credit_card' THEN payment_value ELSE 0 END)::decimal(38, 18) AS credit_card_value,
    SUM(CASE WHEN payment_type = 'boleto' THEN payment_value ELSE 0 END)::decimal(38, 18) AS boleto_value,
    SUM(CASE WHEN payment_type = 'voucher' THEN payment_value ELSE 0 END)::decimal(38, 18) AS voucher_value,
    SUM(CASE WHEN payment_type = 'debit_card' THEN payment_value ELSE 0 END)::decimal(38, 18) AS debit_card_value,

    -- Derived flags
    CASE WHEN max_installments > 1 THEN 1 ELSE 0 END AS is_installment,
    CASE WHEN num_payment_types_used > 1 THEN 1 ELSE 0 END AS is_multi_payment,

    -- Credit card share
    ROUND(try_divide(credit_card_value, total_payment_value) * 100, 5)::decimal(38, 18) AS pct_credit_card
FROM ecommerce.silver.vw_cln_ltst_order_payments
GROUP BY order_id

In [0]:
%sql
select * from ecommerce.silver.vw_payments_agg order by 4 desc

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.silver.vw_enriched_reviews AS

SELECT
    r.review_id,
    r.order_id,
    r.review_score,
    r.review_comment_title,
    r.review_comment_message,
    r.review_creation_date,
    r.review_answer_timestamp,

    -- Review lag from order events
    DATEDIFF(r.review_creation_date, o.order_delivered_customer_date) AS days_to_review_after_delivery,

    round((UNIX_TIMESTAMP(r.review_answer_timestamp) - UNIX_TIMESTAMP(r.review_creation_date)) / 3600, 5)::decimal(38,18) AS review_response_lag_hrs,

    -- Sentiment bucket
    CASE
        WHEN r.review_score >= 4 THEN 'positive'
        WHEN r.review_score =  3 THEN 'neutral'
        ELSE 'negative'
    END AS sentiment,

    -- Flags
    CASE WHEN r.review_comment_message IS NOT NULL THEN 1 ELSE 0 END AS has_comment,

    CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END AS is_low_score

FROM ecommerce.silver.vw_cln_ltst_order_reviews r
LEFT JOIN ecommerce.silver.vw_cln_ltst_orders o ON r.order_id = o.order_id;

In [0]:
%sql
select * from ecommerce.silver.vw_enriched_reviews where order_id = "8f32ee82ea0fe678bd4aae681f1637cc"

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.silver.vw_customer_order_history AS

WITH items_per_order AS (
    SELECT
        order_id,
        COUNT(order_item_id) AS items_in_order,
        SUM(price) AS order_product_value,
        SUM(freight_value) AS order_freight_value,
        SUM(item_total_revenue) AS order_total_item_revenue,
        COUNT(DISTINCT category_english) AS distinct_categories,
        SUM(is_high_value_item) AS high_value_items_count
    FROM ecommerce.silver.vw_order_items_enriched
    GROUP BY order_id
),

orders_full AS (
    -- Combine all order-level silver tables
    SELECT
        o.order_id,
        o.customer_id,
        o.customer_unique_id,
        o.customer_state,
        o.customer_city,
        o.order_purchase_timestamp,
        o.order_purchase_month,
        o.is_delivered,
        o.is_late,
        o.actual_delivery_days,
        o.delivery_delay_days,

        -- Items
        i.items_in_order,
        i.order_product_value,
        i.order_freight_value,
        i.order_total_item_revenue,
        i.distinct_categories,
        i.high_value_items_count,

        -- Payments
        p.total_payment_value,
        p.pct_credit_card,
        p.is_installment,
        p.max_installments,

        -- Reviews
        r.review_score,
        r.sentiment,
        r.has_comment,
        r.is_low_score,
        r.days_to_review_after_delivery

    FROM ecommerce.silver.vw_orders_enriched o
    LEFT JOIN items_per_order i ON o.order_id = i.order_id
    LEFT JOIN ecommerce.silver.vw_payments_agg p ON o.order_id = p.order_id
    LEFT JOIN ecommerce.silver.vw_enriched_reviews r ON o.order_id = r.order_id
)

SELECT
    customer_unique_id,
    max((order_purchase_timestamp, customer_state)).customer_state as customer_state,
    max((order_purchase_timestamp, customer_city)).customer_city as customer_city,

    -- Order behaviour
    COUNT(order_id) AS total_orders,
    SUM(is_delivered) AS delivered_orders,
    SUM(is_late) AS late_orders,
    AVG(actual_delivery_days) AS avg_delivery_days,
    AVG(delivery_delay_days) AS avg_delay_days,

    -- Spend
    SUM(total_payment_value) AS total_spend,
    AVG(total_payment_value) AS avg_order_value,
    MAX(total_payment_value) AS max_order_value,
    SUM(order_freight_value) AS total_freight_paid,

    -- Payment behaviour
    AVG(pct_credit_card) AS avg_pct_credit_card,
    SUM(is_installment) AS installment_orders,
    AVG(max_installments) AS avg_installments,

    -- Product behaviour
    SUM(items_in_order) AS total_items_purchased,
    AVG(items_in_order) AS avg_items_per_order,
    SUM(distinct_categories) AS total_category_touches,
    COUNT(DISTINCT order_purchase_month) AS active_months,

    -- Review behaviour
    AVG(review_score) AS avg_review_score,
    SUM(has_comment) AS reviews_with_comment,
    SUM(is_low_score) AS low_score_reviews,

    -- Recency / Tenure
    MIN(order_purchase_timestamp) AS first_order_date,
    MAX(order_purchase_timestamp) AS last_order_date,

    -- Derived
    DATEDIFF(MAX(order_purchase_timestamp),
    MIN(order_purchase_timestamp)) AS customer_tenure_days,
    ROUND(SUM(is_late) / COUNT(order_id), 4) AS late_order_rate,
    ROUND(try_divide(SUM(total_payment_value), COUNT(DISTINCT order_purchase_month)), 2) AS avg_spend_per_month,
    ROUND(try_divide(SUM(order_freight_value), SUM(total_payment_value)), 4) AS freight_to_spend_ratio

FROM orders_full
GROUP BY
    customer_unique_id

In [0]:
%sql
select * from ecommerce.silver.vw_customer_order_history where customer_unique_id = "f7b62c75467e8ce080b201667cbbc274"